# Project 1: Anomaly Detection in Financial Data
# Using Unsupervised Learning

**Task:** Detect anomalies in financial installment purchase data using unsupervised learning methods.

**Author:** [Your Name] - [Your Student ID]

---


# 1. Load Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_excel('data/Data_finance.xlsx')
print(f"Shape: {df.shape}")
print(f"\nRows: {df.shape[0]}, Columns: {df.shape[1]}")


## 1.1 Calculate INCOME column
Formula: `INCOME = (PERSONAL_INCOME + ADDITIONAL_INCOME + FAMILY_INCOME) - (PERSONAL_EXPENSE + FAMILY_EXPENSE)`


In [ ]:
# Calculate INCOME column
df['INCOME'] = (df['PERSONAL_INCOME'] + df['ADDITIONAL_INCOME'] + df['FAMILY_INCOME']) - \
               (df['PERSONAL_EXPENSE'] + df['FAMILY_EXPENSE'])

print("INCOME column calculated.")
print(f"INCOME - Min: {df['INCOME'].min():,.0f}, Max: {df['INCOME'].max():,.0f}, Mean: {df['INCOME'].mean():,.0f}")


## 1.2 Dataset Information


In [ ]:
df.info()


In [ ]:
# First 5 rows
df.head()


---
# 2. Exploratory Data Analysis & Preprocessing

## 2.1 Descriptive Statistics


In [ ]:
df.describe()


## 2.2 Target Variable Distribution (RESULT_YES/NO)
- 1 = Loan approved (normal)
- 0 = Loan rejected (anomaly)


In [ ]:
print("RESULT_YES/NO Distribution:")
print(df['RESULT_YES/NO'].value_counts())
print(f"\nNormal (1) ratio: {df['RESULT_YES/NO'].mean():.2%}")
print(f"Anomaly (0) ratio: {1 - df['RESULT_YES/NO'].mean():.2%}")

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df['RESULT_YES/NO'].value_counts().plot(kind='bar', color=['coral', 'steelblue'], ax=ax)
ax.set_title('RESULT_YES/NO Distribution')
ax.set_xlabel('Decision')
ax.set_ylabel('Count')
ax.set_xticklabels(['Rejected (0)', 'Approved (1)'], rotation=0)
plt.tight_layout()
plt.show()


## 2.3 Check Missing Values


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_info = pd.DataFrame({'Missing': missing, 'Percent (%)': missing_pct})
print("Columns with missing values:")
print(missing_info[missing_info['Missing'] > 0])


**Observations:**
- `COUNT_TERM_RECEIPT` and `SUM_RECEIPT_AMT` have N/A values → These represent **bad debt** borrowers (borrowed but never repaid).
- `Number of Children` has many missing values → Empty means no children → Fill with 0.
- `Unnamed: 13` is almost entirely missing → Drop it.


## 2.4 Feature Selection

**Columns to drop:**
- `APP_ID`: Just an ID, no analytical value
- `DISBURSALDATE`, `REPORT_DATE`: Date columns, hard to use directly
- `PRODUCT_NAME`: Too many unique values (174), hard to encode effectively
- `City`, `ADDRESS_CURRENT_CITY`, `PROVINCE_CURRENT`: Address info, too many unique values
- `Unnamed: 13`: Almost entirely missing
- `flag term`, `flag loan amt`, `flag final`: Auxiliary columns
- `RESULT_YES/NO`: This is the LABEL, used only for evaluation, NOT for training


In [ ]:
# Columns to drop
cols_to_drop = [
    'APP_ID', 'DISBURSALDATE', 'REPORT_DATE', 'PRODUCT_NAME',
    'City', 'ADDRESS_CURRENT_CITY', 'PROVINCE_CURRENT',
    'Unnamed: 13', 'flag term', 'flag loan amt', 'flag final',
    'RESULT_YES/NO'  # Label - only for evaluation
]

# Save labels before dropping
labels = df['RESULT_YES/NO'].copy()

# Create feature dataframe
df_features = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Remaining columns: {df_features.shape[1]}")
print(f"Columns: {df_features.columns.tolist()}")


## 2.5 Handle Missing Values


In [ ]:
# 1. Number of Children: NaN = no children -> Fill with 0
df_features['Number of Children'] = df_features['Number of Children'].fillna(0)

# 2. COUNT_TERM_RECEIPT & SUM_RECEIPT_AMT: NaN = bad debt -> Fill with 0
df_features['COUNT_TERM_RECEIPT'] = df_features['COUNT_TERM_RECEIPT'].fillna(0)
df_features['SUM_RECEIPT_AMT'] = df_features['SUM_RECEIPT_AMT'].fillna(0)

# Verify
print("Missing values after handling:")
print(df_features.isnull().sum().sum(), "missing values remaining")


## 2.6 Encode Categorical Columns

- **GENDER:** F=0, M=1 (binary encoding)
- **MARITAL_STATUS:** LabelEncoder
- **EDUCATION:** Ordinal encoding by education level (low to high)


In [ ]:
# GENDER: F=0, M=1
df_features['GENDER'] = df_features['GENDER'].map({'F': 0, 'M': 1})
print("GENDER encoded: F=0, M=1")

# EDUCATION: Ordinal encoding (low -> high)
education_order = {
    'No Education': 0, 'Basic School': 1, 'Middle School': 2,
    'High School': 3, 'Working Cerfiticate': 4, 'College': 5,
    'Bachelor': 6, 'Master Degree': 7
}
df_features['EDUCATION'] = df_features['EDUCATION'].map(education_order)
print(f"EDUCATION encoded: {education_order}")

# MARITAL_STATUS: LabelEncoder
le_marital = LabelEncoder()
df_features['MARITAL_STATUS'] = le_marital.fit_transform(df_features['MARITAL_STATUS'])
print(f"MARITAL_STATUS encoded: {dict(zip(le_marital.classes_, le_marital.transform(le_marital.classes_)))}")

print(f"\nShape after encoding: {df_features.shape}")


## 2.7 Feature Normalization

Using StandardScaler to bring all features to the same scale (mean=0, std=1). This is crucial for unsupervised learning algorithms.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

print(f"Shape after scaling: {X_scaled.shape}")
print(f"Mean (should be ~0): {X_scaled.mean(axis=0)[:5].round(4)}")
print(f"Std (should be ~1): {X_scaled.std(axis=0)[:5].round(4)}")


## 2.8 Visualize Feature Distributions


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features_to_plot = ['LA', 'EFF_RATE', 'TERM', 'EMI', 'AGE', 'INCOME']

for i, feat in enumerate(features_to_plot):
    row, col = i // 3, i % 3
    if feat in df_features.columns:
        for val, color, lbl in [(1, 'steelblue', 'Approved'), (0, 'coral', 'Rejected')]:
            axes[row, col].hist(df[df['RESULT_YES/NO'] == val][feat].dropna(),
                                bins=30, alpha=0.6, color=color, label=lbl)
        axes[row, col].set_title(feat)
        axes[row, col].legend()

plt.suptitle('Feature Distributions by Normal / Anomaly', fontsize=14)
plt.tight_layout()
plt.show()


---
# 3. Build Models

## Important Notes
- Only **unsupervised learning** methods are allowed
- `RESULT_YES/NO` is used only for **evaluation**, NOT for training
- Model output: 1 = normal, 0 = anomaly

## 3.1 Model 1: Isolation Forest

**Isolation Forest** detects anomalies based on the idea that anomalous points are easier to "isolate" than normal points.

**Principle:** Build many random decision trees. Data points that get isolated quickly (require fewer splits) are likely anomalies.


In [ ]:
# Anomaly ratio in data
contamination_rate = (labels == 0).sum() / len(labels)
print(f"Actual anomaly ratio: {contamination_rate:.4f} ({contamination_rate:.2%})")

# Train Isolation Forest
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination_rate,
    random_state=42,
    n_jobs=-1
)
iso_pred = iso_forest.fit_predict(X_scaled)

# Convert: Isolation Forest returns 1 (normal) and -1 (anomaly)
iso_pred_binary = np.where(iso_pred == 1, 1, 0)

print(f"\nIsolation Forest Results:")
print(f"  Predicted normal (1): {(iso_pred_binary == 1).sum()}")
print(f"  Predicted anomaly (0): {(iso_pred_binary == 0).sum()}")


### Evaluate Isolation Forest


In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    """Print evaluation metrics for the model."""
    print(f"{'='*50}")
    print(f"MODEL EVALUATION: {model_name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Anomaly (0)', 'Normal (1)']))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Anomaly (0)', 'Normal (1)'],
                yticklabels=['Anomaly (0)', 'Normal (1)'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

evaluate_model(labels, iso_pred_binary, "Isolation Forest")


## 3.2 Model 2: Local Outlier Factor (LOF)

**Local Outlier Factor** measures the local density of each data point compared to its neighbors.

**Principle:** If a point has significantly lower density than its surrounding points, it is likely an anomaly.


In [ ]:
# Train Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination_rate,
    n_jobs=-1
)
lof_pred = lof.fit_predict(X_scaled)

lof_pred_binary = np.where(lof_pred == 1, 1, 0)

print(f"LOF Results:")
print(f"  Predicted normal (1): {(lof_pred_binary == 1).sum()}")
print(f"  Predicted anomaly (0): {(lof_pred_binary == 0).sum()}")

evaluate_model(labels, lof_pred_binary, "Local Outlier Factor (LOF)")


## 3.3 Model 3: One-Class SVM

**One-Class SVM** learns a boundary around "normal" data points.

**Principle:** Find a hyperplane in feature space such that most data lies inside. Points outside the boundary are considered anomalies.


In [ ]:
# Train One-Class SVM
oc_svm = OneClassSVM(
    kernel='rbf',
    gamma='scale',
    nu=contamination_rate
)
svm_pred = oc_svm.fit_predict(X_scaled)

svm_pred_binary = np.where(svm_pred == 1, 1, 0)

print(f"One-Class SVM Results:")
print(f"  Predicted normal (1): {(svm_pred_binary == 1).sum()}")
print(f"  Predicted anomaly (0): {(svm_pred_binary == 0).sum()}")

evaluate_model(labels, svm_pred_binary, "One-Class SVM")


---
# 4. Model Comparison


In [ ]:
results = {
    'Model': ['Isolation Forest', 'Local Outlier Factor', 'One-Class SVM'],
    'Accuracy': [
        accuracy_score(labels, iso_pred_binary),
        accuracy_score(labels, lof_pred_binary),
        accuracy_score(labels, svm_pred_binary)
    ],
    'Precision': [
        precision_score(labels, iso_pred_binary, zero_division=0),
        precision_score(labels, lof_pred_binary, zero_division=0),
        precision_score(labels, svm_pred_binary, zero_division=0)
    ],
    'Recall': [
        recall_score(labels, iso_pred_binary, zero_division=0),
        recall_score(labels, lof_pred_binary, zero_division=0),
        recall_score(labels, svm_pred_binary, zero_division=0)
    ],
    'F1-Score': [
        f1_score(labels, iso_pred_binary, zero_division=0),
        f1_score(labels, lof_pred_binary, zero_division=0),
        f1_score(labels, svm_pred_binary, zero_division=0)
    ]
}

results_df = pd.DataFrame(results)
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results['Model']))
width = 0.2
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['steelblue', 'coral', 'seagreen', 'purple']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i * width, results[metric], width, label=metric, color=color, alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Anomaly Detection Model Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results['Model'])
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()


---
# 5. Conclusion

## Key Findings
- Unsupervised learning methods can detect financial anomalies to a reasonable degree **without labels**.
- **Isolation Forest** typically performs best for anomaly detection tasks as it was specifically designed for this purpose.
- **LOF** works well when anomalies are concentrated in low-density regions.
- **One-Class SVM** finds more complex decision boundaries but is computationally expensive.

## Limitations
- Unsupervised learning does not use labels, so it cannot achieve the same performance as supervised learning.
- The contamination rate needs to be estimated beforehand, which affects results.
- Results heavily depend on feature selection and preprocessing.

## Conclusion
Unsupervised anomaly detection models can provide preliminary credit risk assessment. However, real-world applications would require domain knowledge and more advanced techniques.
